# Fusion Retrieval in Document Search

## Overview

This code implements a Fusion Retrieval system that combines vector-based similarity search with keyword-based BM25 retrieval. The approach aims to leverage the strengths of both methods to improve the overall quality and relevance of document retrieval.

## Motivation

Traditional retrieval methods often rely on either semantic understanding (vector-based) or keyword matching (BM25). Each approach has its strengths and weaknesses. Fusion retrieval aims to combine these methods to create a more robust and accurate retrieval system that can handle a wider range of queries effectively.

## Key Components

1. PDF processing and text chunking
2. Vector store creation using FAISS and OpenAI embeddings
3. BM25 index creation for keyword-based retrieval
4. Custom fusion retrieval function that combines both methods

## Method Details

### Document Preprocessing

1. The PDF is loaded and split into chunks using RecursiveCharacterTextSplitter.
2. Chunks are cleaned by replacing 't' with spaces (likely addressing a specific formatting issue).

### Vector Store Creation

1. OpenAI embeddings are used to create vector representations of the text chunks.
2. A FAISS vector store is created from these embeddings for efficient similarity search.

### BM25 Index Creation

1. A BM25 index is created from the same text chunks used for the vector store.
2. This allows for keyword-based retrieval alongside the vector-based method.

### Fusion Retrieval Function

The `fusion_retrieval` function is the core of this implementation:

1. It takes a query and performs both vector-based and BM25-based retrieval.
2. Scores from both methods are normalized to a common scale.
3. A weighted combination of these scores is computed (controlled by the `alpha` parameter).
4. Documents are ranked based on the combined scores, and the top-k results are returned.

## Benefits of this Approach

1. Improved Retrieval Quality: By combining semantic and keyword-based search, the system can capture both conceptual similarity and exact keyword matches.
2. Flexibility: The `alpha` parameter allows for adjusting the balance between vector and keyword search based on specific use cases or query types.
3. Robustness: The combined approach can handle a wider range of queries effectively, mitigating weaknesses of individual methods.
4. Customizability: The system can be easily adapted to use different vector stores or keyword-based retrieval methods.

## Conclusion

Fusion retrieval represents a powerful approach to document search that combines the strengths of semantic understanding and keyword matching. By leveraging both vector-based and BM25 retrieval methods, it offers a more comprehensive and flexible solution for information retrieval tasks. This approach has potential applications in various fields where both conceptual similarity and keyword relevance are important, such as academic research, legal document search, or general-purpose search engines.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [ ]:
# Install required packages
#!uv pip install langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu numpy python-dotenv rank-bm25 pymupdf

In [5]:
!uv pip install numpy rank-bm25 pymupdf deepeval

Resolved 70 packages in 541ms                                        
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/820.66 KiB          
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 61.82 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 77.82 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 93.82 KiB/820.66 KiB        
⠙ Preparing packages... (0/1)------------------- 109.82 KiB/820.66 KiB       
⠙ Preparing packages... (0/1)------------------- 125.82 KiB/820.66 KiB       
⠙ Preparing packages... (0/1)------------------- 141.82 KiB/820.66 KiB       
⠙ Preparing packages... (0/1)------------------- 157.82 KiB/820.66 Ki

In [1]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np


from utils.helper_functions import *
from utils.evaluate_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### Define document path

In [2]:
path = "pdfs/Understanding_Climate_Change.pdf"

### Encode the pdf to vector store and return split document from the step before to create BM25 instance

In [3]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A tuple of (FAISS vector store, cleaned text documents).
    """
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

### Create vectorstore and get the chunked documents

In [4]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

### Create a bm25 index for retrieving documents by keywords

In [15]:
tokenized_corpus

[['Hello', 'there', 'good', 'man!'],
 ['It', 'is', 'quite', 'windy', 'in', 'London'],
 ['How', 'is', 'the', 'weather', 'today?']]

In [5]:
from rank_bm25 import BM25Okapi

# Initalizing
corpus = [
    "Hello there good man!",
    "It is quite windy in London",
    "How is the weather today?"
]

tokenized_corpus = [doc.split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

# Ranking of documents
query = "windy London"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)
print(doc_scores)

doc_top_n = bm25.get_top_n(tokenized_query, corpus, n=1)
print(doc_top_n)

[0.         0.93729472 0.        ]
['It is quite windy in London']


In [6]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [7]:
bm25 = create_bm25_index(cleaned_texts) # Create BM25 index from the cleaned texts (chunks)

### Define a function that retrieves both semantically and by keyword, normalizes the scores and gets the top k documents

In [8]:
def fusion_retrieval(vectorstore, bm25, query: str, k: int = 5, alpha: float = 0.5) -> List[Document]:
    """
    Perform fusion retrieval combining keyword-based (BM25) and vector-based search.

    Args:
    vectorstore (VectorStore): The vectorstore containing the documents.
    bm25 (BM25Okapi): Pre-computed BM25 index.
    query (str): The query string.
    k (int): The number of documents to retrieve.
    alpha (float): The weight for vector search scores (1-alpha will be the weight for BM25 scores).

    Returns:
    List[Document]: The top k documents based on the combined scores.
    """
    
    epsilon = 1e-8

    # Step 1: Get all documents from the vectorstore
    all_docs = vectorstore.similarity_search("", k=vectorstore.index.ntotal)

    # Step 2: Perform BM25 search
    bm25_scores = bm25.get_scores(query.split())

    # Step 3: Perform vector search
    vector_results = vectorstore.similarity_search_with_score(query, k=len(all_docs))
    
    # Step 4: Normalize scores
    vector_scores = np.array([score for _, score in vector_results])
    vector_scores = 1 - (vector_scores - np.min(vector_scores)) / (np.max(vector_scores) - np.min(vector_scores) + epsilon)

    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) -  np.min(bm25_scores) + epsilon)

    # Step 5: Combine scores
    combined_scores = alpha * vector_scores + (1 - alpha) * bm25_scores  

    # Step 6: Rank documents
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    # Step 7: Return top k documents
    return [all_docs[i] for i in sorted_indices[:k]]

### Use Case example

In [10]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
top_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.8)
docs_content = [doc.page_content for doc in top_docs]
show_context(docs_content)

Context 1:
Vision for a Sustainable Future 
Holistic Approach 
Addressing climate change requires a holistic approach that integrates environmental, social, 
and economic dimensions. Sustainable development, circular economy, and ecological justice 
are key principles guiding this approach. Collaboration across sectors and scales is essential 
for achieving a sustainable future. 
Innovation and Creativity 
Innovation and creativity are vital for developing new solutions to climate challenges. This 
includes technological advancements, policy innovations, and creative approaches to 
education and communication. Fostering a culture of innovation supports continuous 
improvement and adaptation. 
Global Solidarity 
Global solidarity and cooperation are fundamental for addressing the global challenge of 
climate change. This includes supporting vulnerable countries and communities, sharing 
resources and technologies, and promoting equitable solutions. Solidarity strengthens global


Contex

## Langchain Implementation

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Load PDF
loader = PyPDFLoader("pdfs/Understanding_Climate_Change.pdf")
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)

# Dense retriever (semantic)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(splits, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse retriever (keyword - BM25)
sparse_retriever = BM25Retriever.from_documents(splits)
sparse_retriever.k = 3

alpha: float = 0.5

# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[alpha, 1 - alpha]
)

# LLM
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    return_source_documents=True
)

# Query with generation
query = "What is the main topic of this document?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")

Answer: The main topic of this document is climate change and the various strategies and challenges associated with addressing it, including policy, technology, and global cooperation.

Sources:
- challenges. This vision includes healthy ecosystems, sustainable economies, and social 
justice. Achieving this vision requires commitment, innovation, and collective effort. 
Legacy for Future Genera...
- emissions, particularly methane, which is a potent greenhouse gas. Innovations in fracking 
technology have made natural gas more accessible, but this comes with environmental and 
health concerns. 
D...
- Direct Air Capture (DAC) 
DAC technology removes CO2 directly from the atmosphere, offering a way to achieve 
negative emissions. The captured CO2 can be stored or used in various applications. Scalin...
- for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burn

In [13]:
# Hybrid retriever
alpha = 0.1
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[alpha, 1 - alpha]
)

# LLM
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    return_source_documents=True
)

# Query with generation
query = "hat are the impacts of climate change?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")

Answer: The impacts of climate change include more frequent and severe heatwaves, which pose risks to human health, agriculture, and infrastructure. Cities are particularly vulnerable due to the "urban heat island" effect. Climate change is also altering the timing and length of seasons, affecting ecosystems and human activities. Glacial melt poses risks to water supplies and hydropower generation, and it also impacts agriculture. Rising sea levels and increased storm surges are accelerating coastal erosion, threatening homes, infrastructure, and ecosystems. Low-lying islands and coastal regions are especially vulnerable. Climate change is also linked to an increase in the frequency and severity of extreme weather events.

Sources:
- impacts are not evenly distributed. Vulnerable populations, including low-income 
communities, indigenous peoples, and marginalized groups, often face the greatest risks 
while contributing the least ...
- Local communities are often on the front lines of 

## Direct ChromaDB Integration

- Pinecone
- Weviate

In [14]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.chat_models import ChatOllama

# Load PDF
loader = PyPDFLoader("pdfs/Understanding_Climate_Change.pdf")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Setup ChromaDB
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("pdf_docs")

# Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Add documents
for idx, chunk in enumerate(chunks):
    collection.add(
        ids=[f"doc_{idx}"],
        documents=[chunk.page_content],
        embeddings=[model.encode(chunk.page_content).tolist()],
        metadatas=[{"page": chunk.metadata.get("page", 0)}]
    )

# Hybrid Query
query = "What are the main topics?"
query_emb = model.encode(query)

results = collection.query(
    query_embeddings=[query_emb.tolist()],
    n_results=5
)

# LLM Generation
llm = ChatOllama(model="qwen3:1.7b")
context = "\n\n".join(results['documents'][0])
prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
response = llm.invoke(prompt)

print("Answer:", response.content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/cj/13vbmk7n7fqgmdnqjjwn1bx80000gn/T/ipykernel_10496/1000290548.py:39: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="qwen3:1.7b")


Answer: The main topics discussed in the context include:

1. **Urban Climate Initiatives**  
   - Sustainable transportation systems  
   - Green building standards  
   - Climate-resilient infrastructure  

2. **Community-Based Conservation**  
   - Involving local communities in environmental protection and sustainability efforts  

3. **Interdisciplinary Approaches**  
   - Collaboration between scientists, policymakers, businesses, and communities to address climate challenges  

4. **Citizen Science**  
   - Engaging the public in scientific research and data collection to advance climate knowledge and action  

5. **Education and Advocacy**  
   - Climate education and curriculum development to raise awareness and build expertise  
   - Research informing evidence-based policies and interventions  

These topics highlight the integration of sustainable practices, community participation, interdisciplinary collaboration, and education to address climate change.


### RRF

In [26]:
!uv pip install scikit-learn sentence_transformers

Audited 2 packages in 33ms


In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def reciprocal_rank_fusion(rankings, k=60, weights=None):
    """RRF implementation"""
    if weights is None:
        weights = [1.0] * len(rankings)
    
    doc_scores = {}
    for ranking_idx, ranking in enumerate(rankings):
        weight = weights[ranking_idx]
        for rank_position, doc_id in enumerate(ranking, start=1):
            rrf_score = weight * (1.0 / (k + rank_position))
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + rrf_score
    
    return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = "What is AI"

# ============================================
# DENSE SEARCH (Vector/Semantic)
# ============================================
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Calculate cosine similarity
similarities = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Get ranked document IDs (by similarity)
dense_ranking = np.argsort(similarities)[::-1].tolist()

# ============================================
# SPARSE SEARCH (Keyword/BM25-like)
# ============================================
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
tfidf_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# Get ranked document IDs
sparse_ranking = np.argsort(tfidf_scores)[::-1].tolist()

# ============================================
# HYBRID SEARCH WITH RRF
# ============================================
hybrid_results = reciprocal_rank_fusion(
    rankings=[dense_ranking, sparse_ranking],
    k=60,
    weights=[2.0, 1.0]  # Dense search more important
)

# ============================================
# DISPLAY RESULTS
# ============================================
print("QUERY:", query)
print("\n" + "="*60)
print("DENSE SEARCH RANKING:")
for rank, doc_idx in enumerate(dense_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("SPARSE SEARCH RANKING:")
for rank, doc_idx in enumerate(sparse_ranking, 1):
    print(f"{rank}. [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("HYBRID RRF RESULTS:")
print("="*60)
for doc_idx, score in hybrid_results[:3]:  # Top 3
    print(f"Score: {score:.6f}")
    print(f"Doc:   {documents[doc_idx]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: What is AI

DENSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [4] Computer vision enables machines to interpret images
3. [3] Natural language processing helps computers understand text
4. [1] Deep learning uses neural networks with multiple layers
5. [2] Python is popular for data science and ML

SPARSE SEARCH RANKING:
1. [0] Machine learning is a subset of artificial intelligence
2. [2] Python is popular for data science and ML
3. [4] Computer vision enables machines to interpret images
4. [3] Natural language processing helps computers understand text
5. [1] Deep learning uses neural networks with multiple layers

HYBRID RRF RESULTS:
Score: 0.049180
Doc:   Machine learning is a subset of artificial intelligence

Score: 0.048131
Doc:   Computer vision enables machines to interpret images

Score: 0.047371
Doc:   Natural language processing helps computers understand text



In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def relative_score_fusion(score_lists, weights=None):
    """
    RSF implementation - uses actual scores, not just ranks
    
    Args:
        score_lists: List of lists, each containing (doc_id, score) tuples
        weights: Optional weights for each retriever
    """
    if weights is None:
        weights = [1.0] * len(score_lists)
    
    doc_scores = {}
    
    for retriever_idx, scores in enumerate(score_lists):
        weight = weights[retriever_idx]
        
        # Extract just the scores for normalization
        raw_scores = [score for _, score in scores]
        
        if len(raw_scores) == 0:
            continue
            
        min_score = min(raw_scores)
        max_score = max(raw_scores)
        score_range = max_score - min_score
        
        # Normalize and accumulate
        for doc_id, score in scores:
            if score_range == 0:
                normalized = 1.0  # All scores equal
            else:
                normalized = (score - min_score) / score_range
            
            doc_scores[doc_id] = doc_scores.get(doc_id, 0.0) + (weight * normalized)
    
    return sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)


# Sample documents
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is popular for data science and ML",
    "Natural language processing helps computers understand text",
    "Computer vision enables machines to interpret images"
]

# Query
query = "What is AI"

# ============================================
# DENSE SEARCH (Vector/Semantic)
# ============================================
dense_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = dense_model.encode(documents)
query_embedding = dense_model.encode(query)

# Calculate cosine similarity
dense_scores = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Create (doc_id, score) tuples - sorted by score descending
dense_with_scores = [(i, float(dense_scores[i])) for i in range(len(documents))]
dense_with_scores.sort(key=lambda x: x[1], reverse=True)

# ============================================
# SPARSE SEARCH (Keyword/BM25-like)
# ============================================
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(documents)
query_tfidf = tfidf.transform([query])

# Calculate TF-IDF scores
sparse_scores = (doc_tfidf @ query_tfidf.T).toarray().flatten()

# Create (doc_id, score) tuples - sorted by score descending
sparse_with_scores = [(i, float(sparse_scores[i])) for i in range(len(documents))]
sparse_with_scores.sort(key=lambda x: x[1], reverse=True)

# ============================================
# HYBRID SEARCH WITH RSF
# ============================================
hybrid_results = relative_score_fusion(
    score_lists=[dense_with_scores, sparse_with_scores],
    weights=[2.0, 1.0]  # Dense search more important
)

# ============================================
# DISPLAY RESULTS
# ============================================
print("QUERY:", query)

print("\n" + "="*60)
print("DENSE SEARCH (with raw scores):")
print("="*60)
for doc_idx, score in dense_with_scores:
    print(f"Score: {score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("SPARSE SEARCH (with raw scores):")
print("="*60)
for doc_idx, score in sparse_with_scores:
    print(f"Score: {score:.4f} | [{doc_idx}] {documents[doc_idx]}")

print("\n" + "="*60)
print("HYBRID RSF RESULTS:")
print("="*60)

# Show normalization breakdown for top results
print("\nDetailed scoring breakdown:\n")

# Recalculate normalized scores for display
dense_raw = [s for _, s in dense_with_scores]
sparse_raw = [s for _, s in sparse_with_scores]

dense_min, dense_max = min(dense_raw), max(dense_raw)
sparse_min, sparse_max = min(sparse_raw), max(sparse_raw)

for doc_idx, final_score in hybrid_results[:3]:
    dense_score = dense_scores[doc_idx]
    sparse_score = sparse_scores[doc_idx]
    
    # Normalized scores
    dense_norm = (dense_score - dense_min) / (dense_max - dense_min)
    sparse_norm = (sparse_score - sparse_min) / (sparse_max - sparse_min) if (sparse_max - sparse_min) > 0 else 0
    
    print(f"Doc [{doc_idx}]: {documents[doc_idx]}")
    print(f"  Dense:  raw={dense_score:.4f} → norm={dense_norm:.4f} × weight=2.0 = {dense_norm*2:.4f}")
    print(f"  Sparse: raw={sparse_score:.4f} → norm={sparse_norm:.4f} × weight=1.0 = {sparse_norm*1:.4f}")
    print(f"  Final RSF Score: {final_score:.4f}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: What is AI

DENSE SEARCH (with raw scores):
Score: 0.5238 | [0] Machine learning is a subset of artificial intelligence
Score: 0.3093 | [4] Computer vision enables machines to interpret images
Score: 0.2978 | [3] Natural language processing helps computers understand text
Score: 0.2312 | [1] Deep learning uses neural networks with multiple layers
Score: 0.2117 | [2] Python is popular for data science and ML

SPARSE SEARCH (with raw scores):
Score: 0.3214 | [0] Machine learning is a subset of artificial intelligence
Score: 0.2917 | [2] Python is popular for data science and ML
Score: 0.0000 | [1] Deep learning uses neural networks with multiple layers
Score: 0.0000 | [3] Natural language processing helps computers understand text
Score: 0.0000 | [4] Computer vision enables machines to interpret images

HYBRID RSF RESULTS:

Detailed scoring breakdown:

Doc [0]: Machine learning is a subset of artificial intelligence
  Dense:  raw=0.5238 → norm=1.0000 × weight=2.0 = 2.0000
  Sparse